# Forward and backward stepwise modeling



## Setup


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from palmerpenguins import load_penguins


## Data


In [ ]:
penguins = load_penguins().dropna()
penguins.columns

## Regression

### Data Prep

In [ ]:
mass_features = ["bill_length_mm", "bill_depth_mm", "species", 
                 "flipper_length_mm", "island", "sex"]
X_mass = (
    pd.get_dummies(penguins[mass_features], dtype = int, drop_first=True)
) 
y_mass = penguins["body_mass_g"]



X_train, X_test, y_train, y_test = train_test_split(
    X_mass, y_mass,
    test_size=0.20,
    random_state=42
)

### Cross-Validation Setup

In [ ]:
cv = KFold(
    n_splits = 10, 
    shuffle=True, 
    random_state=909
    )

### Forward Selection Complete

In [ ]:

results = []
mse = []
p = X_train.shape[1]

for i in range(1, p):
    selector = SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",               
        n_features_to_select=i,
        scoring="neg_mean_squared_error",
        cv=cv,
    )
    selector.fit(X_train, y_train)

    X_selected = selector.transform(X_train)
    scores = cross_val_score(
        estimator=selector.estimator,
        X=X_selected,
        y=y_train,
        scoring=selector.scoring,
        cv=selector.cv,
        n_jobs=-1
    )
    mean_mse = -scores.mean()

    results.append({
        "n_features": i,
        "features": X_train.columns[selector.get_support()],
        "cv_mse": mean_mse
    })
    mse.append(mean_mse)
    print(i)



In [ ]:
mse_vals = np.array(mse)
min = np.argmin(mse_vals)
print(min)

In [ ]:
results[min]['features']

### Fit Model

In [ ]:
XX_train = X_train[results[min]['features']]
final_model= LinearRegression()
final_model.fit(XX_train, y_train)

### Prediction Error

In [ ]:
XX_test = X_test[results[min]['features']]
y_pred = final_model.predict(XX_test)
test_mse = mean_squared_error(y_test, y_pred)
print(np.sqrt(test_mse))

## Classification

### Data Prep

In [ ]:
species_features = ["bill_length_mm", "bill_depth_mm", "body_mass_g", 
                 "flipper_length_mm"]
X_species = penguins[species_features]
y_species = pd.Categorical(penguins["species"])

X_train, X_test, y_train, y_test = train_test_split(
    X_species, y_species,
    test_size=0.20,
    random_state=42
)

### Cross Validation Set Up

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=909)

### Backward Selection Set Up

In [ ]:

results = []
mse = []
p = X_train.shape[1]

for i in range(1, p):
    selector = SequentialFeatureSelector(
        estimator=LogisticRegression(max_iter = 1000),
        direction="backward",               
        n_features_to_select=i,
        scoring="accuracy",
        cv=cv,
    )
    selector.fit(X_train, y_train)

    X_selected = selector.transform(X_train)
    scores = cross_val_score(
        estimator=selector.estimator,
        X=X_selected,
        y=y_train,
        scoring=selector.scoring,
        cv=selector.cv,
        n_jobs=-1
    )
    mean_mse = scores.mean()

    results.append({
        "n_features": i,
        "features": X_train.columns[selector.get_support()],
        "cv_mse": mean_mse
    })
    mse.append(mean_mse)
    print(i)



### Selecting Predictors

In [ ]:
mse_vals = np.array(mse)
min = np.argmax(mse_vals)
print(min)

### Fitting Model

In [ ]:
XX_train = X_train[results[min]['features']]
final_model= LogisticRegression(max_iter=1000)
final_model.fit(XX_train, y_train)

### Fitted Model

In [ ]:
XX_test = X_test[results[min]['features']]
y_pred = final_model.predict(XX_test)
print(accuracy_score(y_test, y_pred))